                                          Uber Eats Bangalore Restaurant dataset                                                      

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv(r"C:\Guvi\first project\Uber_Eats_data.csv")

In [3]:
data.columns

Index(['name', 'online_order', 'book_table', 'rate', 'votes', 'phone',
       'location', 'rest_type', 'dish_liked', 'cuisines',
       'approx_cost(for two people)', 'listed_in(type)', 'listed_in(city)'],
      dtype='object')

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23193 entries, 0 to 23192
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   name                         23193 non-null  object
 1   online_order                 23193 non-null  object
 2   book_table                   23193 non-null  object
 3   rate                         23193 non-null  object
 4   votes                        23193 non-null  int64 
 5   phone                        23193 non-null  object
 6   location                     23193 non-null  object
 7   rest_type                    23193 non-null  object
 8   dish_liked                   23193 non-null  object
 9   cuisines                     23193 non-null  object
 10  approx_cost(for two people)  23193 non-null  object
 11  listed_in(type)              23193 non-null  object
 12  listed_in(city)              23193 non-null  object
dtypes: int64(1), object(12)
memory 

In [5]:
data.shape

(23193, 13)

In [6]:
data["name"]

0                                                    Jalsa
1                                           Spice Elephant
2                                          San Churro Cafe
3                                    Addhuri Udupi Bhojana
4                                            Grand Village
                               ...                        
23188                                   Izakaya Gastro Pub
23189          M Bar - Bengaluru Marriott Hotel Whitefield
23190                               Keys Cafe - Keys Hotel
23191                                              Bhagini
23192    Chime - Sheraton Grand Bengaluru Whitefield Ho...
Name: name, Length: 23193, dtype: object

In [7]:
data["online_order"]

0        Yes
1        Yes
2        Yes
3         No
4         No
        ... 
23188    Yes
23189     No
23190     No
23191     No
23192     No
Name: online_order, Length: 23193, dtype: object

data cleaning


In [8]:
data['name'] = data['name'].str.strip()

encoding

In [9]:
data['online_order'] = data['online_order'].map({'Yes': 1, 'No': 0})
data['book_table'] = data['book_table'].map({'Yes': 1, 'No': 0})

In [10]:
data['rate'] = data['rate'].str.split('/').str[0]
data['rate'] = data['rate'].replace(['NEW', '-'], None)
data['rate'] = pd.to_numeric(data['rate'], errors='coerce')

In [11]:
data['votes'] = pd.to_numeric(data['votes'], errors='coerce')

In [12]:
data.drop(columns=['phone', 'dish_liked'], inplace=True, errors='ignore')

In [13]:
data['location'] = data['location'].str.strip()
data['rest_type'] = data['rest_type'].fillna('Unknown').str.strip()
data['cuisines'] = data['cuisines'].fillna('Unknown').str.strip()
data['listed_in(type)'] = data['listed_in(type)'].str.strip()

In [14]:
data['approx_cost(for two people)'] = data['approx_cost(for two people)'].str.replace(',', '')
data['approx_cost(for two people)'] = pd.to_numeric(
    data['approx_cost(for two people)'], errors='coerce'
)

In [15]:
data.drop(columns=['listed_in(city)'], inplace=True)

In [16]:
data.drop_duplicates(inplace=True)

In [17]:
print(data.info())
print(data.head())

<class 'pandas.core.frame.DataFrame'>
Index: 16619 entries, 0 to 23192
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   name                         16619 non-null  object 
 1   online_order                 16619 non-null  int64  
 2   book_table                   16619 non-null  int64  
 3   rate                         16535 non-null  float64
 4   votes                        16619 non-null  int64  
 5   location                     16619 non-null  object 
 6   rest_type                    16619 non-null  object 
 7   cuisines                     16619 non-null  object 
 8   approx_cost(for two people)  16619 non-null  int64  
 9   listed_in(type)              16619 non-null  object 
dtypes: float64(1), int64(4), object(5)
memory usage: 1.4+ MB
None
                    name  online_order  book_table  rate  votes      location  \
0                  Jalsa             1           1  

In [18]:
bins = [0,400,800,1500,10000]
labels = ['Low','Mid','Premium','Luxury']
data['price_segment'] = pd.cut(data['approx_cost(for two people)'], bins=bins, labels=labels)

In [19]:
data['rating_category'] = pd.cut(
    data['rate'],
    bins=[0,3,4,5],
    labels=['Low','Average','High']
)

sql


In [20]:
import sqlite3

In [21]:
conn = sqlite3.connect(r'C:\Guvi\first project\uber_eats.db')
cursor = conn.cursor()

In [22]:
data.to_sql('restaurants', conn, if_exists='replace', index=False)

16619

In [23]:
pd.read_sql("SELECT * FROM restaurants LIMIT 5", conn)

,name,online_order,book_table,rate,votes,location,rest_type,cuisines,approx_cost(for two people),listed_in(type),price_segment,rating_category
0,Jalsa,1,1,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800,Buffet,Mid,High
1,Spice Elephant,1,0,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800,Buffet,Mid,High
2,San Churro Cafe,1,0,3.8,918,Banashankari,"Cafe, Casual Dining","Cafe, Mexican, Italian",800,Buffet,Mid,Average
3,Addhuri Udupi Bhojana,0,0,3.7,88,Banashankari,Quick Bites,"South Indian, North Indian",300,Buffet,Low,Average
4,Grand Village,0,0,3.8,166,Basavanagudi,Casual Dining,"North Indian, Rajasthani",600,Buffet,Mid,Average


In [24]:
def run_query(query):
    return pd.read_sql(query, conn)

In [25]:
# 1Highest-rated locations
query = """
SELECT location, ROUND(AVG(rate),2) AS avg_rating, COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY location
ORDER BY avg_rating DESC
LIMIT 10;
"""
run_query(query)


,location,avg_rating,total_restaurants
0,Koramangala 5th Block,4.22,1217
1,Lavelle Road,4.21,352
2,Cunningham Road,4.17,207
3,St. Marks Road,4.16,217
4,Koramangala 4th Block,4.13,401
5,Koramangala 3rd Block,4.13,94
6,Sankey Road,4.11,15
7,Richmond Road,4.11,199
8,Residency Road,4.11,306
9,Church Street,4.11,408


In [26]:
#2Over-saturated locations
query = """
SELECT location, COUNT(*) AS restaurant_count
FROM restaurants
GROUP BY location
ORDER BY restaurant_count DESC
LIMIT 10;
"""
run_query(query)

,location,restaurant_count
0,Koramangala 5th Block,1217
1,Indiranagar,1185
2,HSR,842
3,BTM,807
4,Whitefield,781
5,Jayanagar,697
6,JP Nagar,693
7,Marathahalli,631
8,Sarjapur Road,413
9,MG Road,413


In [27]:
#3Online ordering impact
query = """
SELECT online_order, ROUND(AVG(rate),2) AS avg_rating
FROM restaurants
GROUP BY online_order;
"""
run_query(query)

,online_order,avg_rating
0,0,3.99
1,1,3.92


In [28]:
#4Table booking impact
query = """
SELECT book_table, ROUND(AVG(rate),2) AS avg_rating
FROM restaurants
GROUP BY book_table;
"""
run_query(query)

,book_table,avg_rating
0,0,3.84
1,1,4.19


In [29]:
#5Price segment performance
query = """
SELECT price_segment, ROUND(AVG(rate),2) AS avg_rating, COUNT(*) AS total
FROM restaurants
GROUP BY price_segment
ORDER BY avg_rating DESC;
"""
run_query(query)

,price_segment,avg_rating,total
0,Luxury,4.26,1283
1,Premium,4.13,4134
2,Mid,3.85,6760
3,Low,3.83,4442


In [30]:
#6Most common cuisines
query = """
SELECT cuisines, COUNT(*) AS count
FROM restaurants
GROUP BY cuisines
ORDER BY count DESC
LIMIT 10;
"""
run_query(query)

,cuisines,count
0,North Indian,810
1,"North Indian, Chinese",525
2,South Indian,254
3,Cafe,166
4,"Bakery, Desserts",156
5,"Ice Cream, Desserts",148
6,"South Indian, North Indian, Chinese",142
7,Chinese,139
8,"Desserts, Beverages",137
9,Desserts,123


In [31]:
#7Cost vs rating
query = """
SELECT
    price_segment AS price_range,
    ROUND(AVG(rate),2) AS avg_rating
FROM restaurants
GROUP BY price_segment
ORDER BY avg_rating DESC;
"""
run_query(query)


,price_range,avg_rating
0,Luxury,4.26
1,Premium,4.13
2,Mid,3.85
3,Low,3.83


In [32]:
#8Best locations for premium restaurants
query = """
SELECT location, ROUND(AVG(rate),2) AS avg_rating
FROM restaurants
WHERE price_segment IN ('Premium','Luxury')
GROUP BY location
ORDER BY avg_rating DESC
LIMIT 10;
"""
run_query(query)

,location,avg_rating
0,St. Marks Road,4.37
1,Koramangala 7th Block,4.34
2,Koramangala 1st Block,4.34
3,Koramangala 5th Block,4.33
4,Cunningham Road,4.31
5,Sarjapur Road,4.28
6,Koramangala 4th Block,4.24
7,Malleshwaram,4.23
8,Lavelle Road,4.23
9,Shanti Nagar,4.21


In [33]:
#9High demand but low rating areas
query = """
SELECT location,
       COUNT(*) AS total_restaurants,
       ROUND(AVG(rate),2) AS avg_rating
FROM restaurants
GROUP BY location
HAVING total_restaurants > 500 AND avg_rating < 3.8
ORDER BY total_restaurants DESC;
"""
run_query(query)


,location,total_restaurants,avg_rating
0,Marathahalli,631,3.75


In [34]:
#10Combined feature impact
query = """
SELECT online_order, book_table, ROUND(AVG(rate),2) AS avg_rating
FROM restaurants
GROUP BY online_order, book_table
ORDER BY avg_rating DESC;
"""
run_query(query)

,online_order,book_table,avg_rating
0,0,1,4.20
1,1,1,4.17
2,0,0,3.84
3,1,0,3.84
